---
title: "Building Specialized Agents with LangGraph"
draft: true
image: "./img/course-banner.png"
description: "Design a stateful, reviewable agent workflow for evidence-backed technical briefs without rebuilding a general-purpose agent harness."
categories: [agents, workflows, langgraph]
---

An ordinary model API, a few typed tools, and a small Python loop are enough to begin building an agent. This course studies the cases where a domain workflow needs explicit state, branching, parallel work, review gates, durable checkpoints, and a falsifiable completion contract. It is self-contained: the general agent-harness course described in `todos/agent-harness-plan.md` is a planned companion course, not a prerequisite.

The running project is an **evidence brief agent**. Given a technical question and a bounded source set, it plans an investigation, collects and extracts evidence, reconciles conflicting claims, drafts a brief with citations, pauses for review, and produces a versioned final artifact. The domain logic is kept in tools, schemas, and skills. LangGraph owns the workflow state and control flow.

**Design rule:** use a graph when the workflow has state transitions that deserve names, checkpoints, or review; keep ordinary domain knowledge outside the graph.

## What this course covers

The course deliberately does not rebuild every part of a generic coding-agent harness. The planned agent-harness course covers model clients, tool protocols, the agent loop, coding tools, context management, permissions, hooks, sub-agents, MCP, and trajectory evaluation. Those topics are useful background, but this course introduces the small model-and-tool adapter it needs and then concentrates on domain workflow orchestration. The software-engineering plan covers turning a finished harness into a shipped product.

The focus is the orchestration layer:

| Phase | Chapters | Question answered |
|---|---:|---|
| Foundations | 01–02 | Is this task a graph problem, and what state must be explicit? |
| Evidence workflow | 03–05 | How do tools, routing, retries, and parallel branches produce an auditable result? |
| Governance | 06–08 | How can a person review, resume, inspect, and evaluate the workflow? |
| Capstone | 09 | Does the complete system produce useful briefs under adversarial and failure cases? |

## Learning path

| Chapter | Main build | Main experiment |
|---|---|---|
| 01. Where LangGraph belongs | A task-boundary test and a plain-Python baseline | Compare a one-shot agent, a skill-driven harness, and an explicit graph |
| 02. State and graph control | `StateGraph`, typed state, reducers, edges, and `Command` | Expose hidden state and illegal transitions in a linear prototype |
| 03. Tools, evidence, and domain skills | Source, retrieval, extraction, and citation contracts | Measure unsupported claims when contracts are weakened |
| 04. Routing, validation, and recovery | Conditional routes, structured validation, retries, and bounded loops | Compare blind retries with typed recovery routes |
| 05. Parallel research and fan-in | `Send`-based map-reduce and evidence reconciliation | Measure coverage, contradiction rate, latency, and cost under fan-out |
| 06. Review gates and interrupts | `interrupt()`, approval payloads, edits, and rejection paths | Test whether review changes the final brief rather than merely delaying it |
| 07. Persistence, replay, and memory | Checkpointers, `thread_id`, state history, and subgraphs | Resume after failure and fork alternative decisions from a checkpoint |
| 08. Evaluation and operational contracts | Trace records, invariants, budgets, and regression suites | Separate source quality, reasoning quality, workflow quality, and artifact quality |
| 09. Capstone | The complete evidence brief agent | Produce a scorecard against held-out questions and controlled failures |

## Capstone project: Evidence Brief Agent

The capstone accepts a question such as “Should this team adopt a new vector database for a regulated search service?” It must return a brief containing a scoped recommendation, cited claims, uncertainty notes, contradictory evidence, and a review record. A run is not complete merely because a language model returned text.

The graph must expose at least these stages:

```text
intake -> plan -> collect ─┬─> extract -> reconcile -> draft -> review
                           └─> gap check ────────────────┘       |
                                                               ├─ approve -> verify -> export
                                                               └─ revise  -> draft
```

The implementation must use a typed state schema, conditional routing, bounded revision, parallel collection, a checkpointer, and at least one human interrupt. It must preserve source identifiers through extraction and drafting so every material claim can be traced to evidence. It must also provide a deterministic fixture mode so the evaluation suite does not depend on live web availability.

### Completion standard

- Every final claim has a source reference or is explicitly labeled as an inference, assumption, or unresolved question.
- A contradictory source causes a visible reconciliation decision, not silent omission.
- A reviewer can reject or edit a draft and resume the same thread without rerunning completed collection work.
- A crashed or deliberately failed node resumes from a checkpoint without duplicating non-idempotent side effects.
- Revision loops terminate under a fixed budget.
- The evaluation report separates task success, evidence coverage, citation correctness, review compliance, latency, and cost.
- An ablation compares the graph workflow with a simpler skill-driven baseline.

## Prerequisites

- Python, type hints, pytest, and basic async programming.
- Basic model APIs: messages, structured outputs, and tool calls. The first chapter supplies the minimal adapter used by later chapters.
- Basic information-retrieval concepts: queries, documents, passages, claims, and citations.
- No prior LangGraph experience. The course uses the low-level graph API first and treats higher-level agent helpers as optional integrations.
- No prior agent-harness course. The planned harness course can be taken separately or afterward for a deeper treatment of generic agent runtimes.

The practical stack is Python, `langgraph`, a model adapter, a deterministic fixture corpus, pytest, and an in-memory checkpointer during development. A durable database-backed checkpointer is discussed only after the state model and recovery behavior are tested.

## Important boundary

Skills remain the right place for reusable domain instructions, source-handling rules, writing conventions, and tool guidance. LangGraph becomes useful when those instructions need to participate in a named, inspectable, resumable workflow. The capstone therefore keeps a `domain/` layer separate from the graph definition and tests both boundaries independently. The planned agent-harness course can later supply a richer runtime beneath the same graph boundary.

## Reference documentation

- [LangGraph overview](https://docs.langchain.com/oss/python/langgraph/overview)
- [Graph API](https://docs.langchain.com/oss/python/langgraph/use-graph-api)
- [Persistence](https://docs.langchain.com/oss/python/langgraph/persistence)
- [Interrupts](https://docs.langchain.com/oss/python/langgraph/interrupts)
- [Subgraphs](https://docs.langchain.com/oss/python/langgraph/use-subgraphs)
